##**Step 1**: Install Spark

###**Step 2**: Setting up Spark

Before you can connect to a Spark cluster, Spark needs to be installed. The code below is boilerplate code that can be used to set-up Spark. Please note that this code will be leveraged in all the notebooks since each nodebook is a separate entity.

### **Step 3**. Import the lib

In [1]:
# Colab-friendly setup for PySpark
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!pip install -q "pyspark[connect]==3.5.1" "dataproc-spark-connect==0.8.3"

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("RDDPractice").getOrCreate()
sc = spark.sparkContext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 16.1 MB/s eta 0:00:00


## **Step 4** Refering to Demo RDD

> Demo 1

> Demo 2

> Demo 3

> Demo 4

> Demo 5

> Demo 6

> Demo 7

> Demo 8

> Demo 9

> Demo 10

> Demo 11

> Demo 12

> Demo 13

> Demo 14

> Demo 15

> Demo 16

> Demo 17

> Demo 18

> Demo 19

> Demo 20





In [2]:
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
# 1. create an RDD by using parallelize()
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
parallelrdd = sc.parallelize([0, 1, 2, 3, 4, 5, 6, 7, 8])
print(parallelrdd)
# notice the type of RDD created:
# ParallelCollectionRDD[0] at parallelize at PythonRDD.scala:423

print(parallelrdd.count())
# this action will return 9 as this is the number of elements in our collection

print(parallelrdd.collect())
# will return the parallel collection as a list as follows:
# [0, 1, 2, 3, 4, 5, 6, 7, 8]



ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:289
9
[0, 1, 2, 3, 4, 5, 6, 7, 8]


In [3]:
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
# 2. range()
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
# create an RDD with 1000 integers starting at 0 in increments of 1
# across 2 partitions
range_rdd = sc.range(0, 1000, 1, 2)
print(range_rdd)
# note the PythonRDD type, as range is a native Python function
# PythonRDD[1] at RDD at PythonRDD.scala:43

print(range_rdd.getNumPartitions())
# should return 2 as we requested numSlices=2 range_rdd.min()
# should return 0 as this was out start argument

print(range_rdd.max())
# should return 999 as this is 1000 increments of 1 starting from 0

print(range_rdd.take(5))
# should return [0, 1, 2, 3, 4]


PythonRDD[3] at RDD at PythonRDD.scala:53
2
999
[0, 1, 2, 3, 4]


In [4]:
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
# 3. create empty RDD
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
emptyRDD  = sc.emptyRDD()
print("is Empty RDD  : " + str(emptyRDD.isEmpty()))
emptyRDD2 = sc.parallelize([])
print("is Empty RDD2 : " + str(emptyRDD2.isEmpty()))



is Empty RDD  : True
is Empty RDD2 : True


In [5]:
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
# 6. Create an RDD using parallelize() or range()
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
# Create an RDD of integers
numbers = sc.range(0, 1_000_000, 1, numSlices=2)

# ✅ Correct filter for even numbers (x % 2 == 0)
evens = numbers.filter(lambda x: x % 2 == 0)

# Count elements in the filtered RDD
noelements = evens.count()
print(f"There are {noelements:,} even elements in the collection")

# Collect only a small subset to avoid reprocessing the full dataset
listofelements = evens.take(5)
print(f"The first five even numbers are: {listofelements}")


There are 500,000 even elements in the collection
The first five even numbers are: [0, 2, 4, 6, 8]


In [6]:
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
# 7. create an JOIN by using parallelize()
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
stores = sc.parallelize([
   (100, 'Boca Raton'),
   (101, 'Columbia'),
   (102, 'Cambridge'),
   (103, 'Naperville')
])
# stores schema (store_id, store_location)


In [7]:
salespeople = sc.parallelize([
   (1, 'Henry', 100),
   (2, 'Karen', 100),
   (3, 'Paul', 101),
   (4, 'Jimmy', 102),
   (5, 'Janice', None)
])
# salespeople schema (salesperson_id, salesperson_name, store_id)

In [8]:
print(salespeople.keyBy(lambda x: x[2]) \
                 .join(stores).collect())
# returns:
# [
#  (100, ((1, 'Henry', 100), 'Boca Raton')),
#  (100, ((2, 'Karen', 100), 'Boca Raton')),
#  (102, ((4, 'Jimmy', 102), 'Cambridge')),
#  (101, ((3, 'Paul', 101), 'Columbia'))
# ]


[(100, ((1, 'Henry', 100), 'Boca Raton')), (100, ((2, 'Karen', 100), 'Boca Raton')), (101, ((3, 'Paul', 101), 'Columbia')), (102, ((4, 'Jimmy', 102), 'Cambridge'))]


In [9]:
print(salespeople.keyBy(lambda x: x[2]) \
                 .leftOuterJoin(stores) \
                 .filter(lambda x: x[1][1] is None) \
                 .map(lambda x: "salesperson " + x[1][0][1] + " has no store") \
                 .collect())
# returns ['salesperson Janice has no store']


['salesperson Janice has no store']


In [10]:
print(salespeople.keyBy(lambda x: x[2]) \
                 .rightOuterJoin(stores) \
                 .filter(lambda x: x[1][0] is None) \
                 .map(lambda x: x[1][1] + " store has no salespeople") \
                 .collect() )
# returns ['Naperville store has no salespeople']

['Naperville store has no salespeople']


In [11]:
print(salespeople.keyBy(lambda x: x[2]) \
                 .fullOuterJoin(stores) \
                 .filter(lambda x: x[1][0] is None or x[1][1] is None) \
                 .collect() )
# returns [(,([5,'Janice',], None)),(103,(None,[103,'Naperville']))]

[(None, ((5, 'Janice', None), None)), (103, (None, 'Naperville'))]


In [12]:
print(salespeople.keyBy(lambda x: x[2]) \
                 .cogroup(stores).take(1))
# returns:
# [(None, (<pyspark.resultiterable.ResultIterable object at ...>,
# <pyspark.resultiterable.ResultIterable object at ...>))]


[(100, (<pyspark.resultiterable.ResultIterable object at 0x7c5a94397140>, <pyspark.resultiterable.ResultIterable object at 0x7c5a941fdaf0>))]


In [13]:
print(salespeople.keyBy(lambda x: x[2]) \
                 .cogroup(stores) \
                 .mapValues(lambda x: [item for sublist in x for item in sublist]) \
                 .collect())
# using the mapValues() to process the Iterable object returns:
# [(None, [(5, 'Janice', None)]),
# (100, [(1, 'Henry', 100), (2, 'Karen', 100), 'Boca Raton']),
# (102, [(4, 'Jimmy', 102), 'Cambridge']), (101, [(3, 'Paul', 101),'Columbia']),
# (103, ['Naperville'])]


[(100, [(1, 'Henry', 100), (2, 'Karen', 100), 'Boca Raton']), (None, [(5, 'Janice', None)]), (101, [(3, 'Paul', 101), 'Columbia']), (102, [(4, 'Jimmy', 102), 'Cambridge']), (103, ['Naperville'])]


In [14]:

print(salespeople.keyBy(lambda x: x[2]) \
                 .cartesian(stores).take(1))
# returns:
# [((100, (1, 'Henry', 100)), (100, 'Boca Raton'))]

print(salespeople.keyBy(lambda x: x[2]) \
                 .cartesian(stores).count())
# returns 20 as there are 5 x 4 = 20 records



[((100, (1, 'Henry', 100)), (100, 'Boca Raton'))]
20


In [15]:
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
# 8. create an UNION by using parallelize()
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
odds = sc.parallelize([1,3,5,7,9])
fibonacci = sc.parallelize([0,1,2,3,5,8])
print(odds.union(fibonacci).collect())
# returns [1, 3, 5, 7, 9, 0, 1, 2, 3, 5, 8]

[1, 3, 5, 7, 9, 0, 1, 2, 3, 5, 8]


In [ ]:
odds = sc.parallelize([1,3,5,7,9])
fibonacci = sc.parallelize([0,1,2,3,5,8])
print(odds.intersection(fibonacci).collect())
# returns [1, 3, 5]


In [ ]:
odds = sc.parallelize([1,3,5,7,9])
fibonacci = sc.parallelize([0,1,2,3,5,8])
print(odds.subtract(fibonacci).collect())
# returns [7, 9]


[9, 7]


In [ ]:
cities1 = sc.parallelize(
[
   ('Hayward',(37.668819,-122.080795)),
   ('Baumholder',(49.6489,7.3975)),
   ('Alexandria',(38.820450,-77.050552)),
   ('Melbourne', (37.663712,144.844788))
])

In [ ]:
cities2 = sc.parallelize(
[
   ('Boulder Creek',(64.0708333,-148.2236111)),
   ('Hayward',(37.668819,-122.080795)),
   ('Alexandria',(38.820450,-77.050552)),
   ('Arlington', (38.878337,-77.100703))
])
print(cities1.subtractByKey(cities2).collect())
# returns:
# [('Baumholder', (49.6489, 7.3975)), ('Melbourne', (37.663712,144.844788))]


[('Baumholder', (49.6489, 7.3975)), ('Melbourne', (37.663712, 144.844788))]


In [ ]:
print(cities2.subtractByKey(cities1).collect())
# returns:
# [('Boulder Creek', (64.0708333, -148.2236111)),
# ('Arlington', (38.878337, -77.100703))]


[('Arlington', (38.878337, -77.100703)), ('Boulder Creek', (64.0708333, -148.2236111))]


In [ ]:
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
# 8. create an STATISTICS by using parallelize()
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
numbers = sc.parallelize([0,1,1,2,3,5,8,13,21,34])
print(numbers.min())
# returns 0

0


In [ ]:
numbers = sc.parallelize([0,1,1,2,3,5,8,13,21,34])
print(numbers.max())
# returns 34


34


In [ ]:
numbers = sc.parallelize([0,1,1,2,3,5,8,13,21,34])
print(numbers.mean())
# returns 8.8


8.8


In [ ]:
numbers = sc.parallelize([0,1,1,2,3,5,8,13,21,34])
print(numbers.sum())
# returns 88

88


In [ ]:

numbers = sc.parallelize([0,1,1,2,3,5,8,13,21,34])
numbers.stdev()
# returns 10.467091286503619

np.float64(10.467091286503619)

In [ ]:

numbers = sc.parallelize([0,1,1,2,3,5,8,13,21,34])
print(numbers.variance())
# returns 109.55999999999999

109.55999999999999


In [ ]:
numbers = sc.parallelize([0,1,1,2,3,5,8,13,21,34])
numbers.stats()
# returns (count: 10, mean: 8.8, stdev: 10.4670912865, max: 34.0, min:0.0)

(count: 10, mean: 8.8, stdev: 10.467091286503619, max: 34.0, min: 0.0)

## **Step4** create an SQL by using parallelize()

In [ ]:
## mount to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
# 8. create an SQL by using parallelize()
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
spark = (SparkSession
   .builder
   .appName("SparkSQLExampleApp")
   .getOrCreate())


In [ ]:
# Path to data set
csv_file = "/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Dataset/airline.csv"

In [ ]:
import pandas as pd
df = pd.read_csv(csv_file)
df.head()

,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,...,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2005,1,1,6,1211.0,1216,1451.0,1502,UA,548,...,2.0,17.0,0,NaN,0,0.0,0.0,0.0,0.0,0.0
1,2005,1,2,7,1209.0,1216,1447.0,1502,UA,548,...,2.0,17.0,0,NaN,0,0.0,0.0,0.0,0.0,0.0
2,2005,1,3,1,1213.0,1216,1454.0,1502,UA,548,...,3.0,15.0,0,NaN,0,0.0,0.0,0.0,0.0,0.0
3,2005,1,4,2,NaN,1216,NaN,1502,UA,548,...,0.0,0.0,1,A,0,0.0,0.0,0.0,0.0,0.0
4,2005,1,5,3,1211.0,1216,1504.0,1502,UA,548,...,6.0,22.0,0,NaN,0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Read and create a temporary view
# Infer schema (note that for larger files you
# may want to specify the schema)
df = (spark.read.format("csv")
   .option("inferSchema", "true")
   .option("header", "true")
   .load(csv_file))
df.createOrReplaceTempView("us_delay_flights_tbl")


In [ ]:
print("find all flights whose distance is greater than 1,000 miles: ")

spark.sql("""SELECT Distance, Origin, Dest
FROM us_delay_flights_tbl WHERE Distance > 1000
ORDER BY Distance DESC""").show(10)

find all flights whose distance is greater than 1,000 miles: 
+--------+------+----+
|Distance|Origin|Dest|
+--------+------+----+
|    2704|   SFO| BOS|
|    2704|   SFO| BOS|
|    2704|   SFO| BOS|
|    2704|   SFO| BOS|
|    2704|   SFO| BOS|
|    2704|   SFO| BOS|
|    2704|   SFO| BOS|
|    2704|   SFO| BOS|
|    2704|   SFO| BOS|
|    2704|   SFO| BOS|
+--------+------+----+
only showing top 10 rows



In [ ]:
print("find all flights origin from JFK: ")

df_jfk = spark.sql("""
SELECT DayOfWeek, DepDelay, Origin, Dest FROM
us_delay_flights_tbl WHERE origin = 'SFO'""").show(10)


find all flights origin from JFK: 
+---------+--------+------+----+
|DayOfWeek|DepDelay|Origin|Dest|
+---------+--------+------+----+
|        6|      -5|   SFO| SLC|
|        7|      -7|   SFO| SLC|
|        1|      -3|   SFO| SLC|
|        2|      NA|   SFO| SLC|
|        3|      -5|   SFO| SLC|
|        4|      -1|   SFO| SLC|
|        5|      75|   SFO| SLC|
|        6|      -2|   SFO| SLC|
|        7|      -9|   SFO| SLC|
|        1|      83|   SFO| SLC|
+---------+--------+------+----+
only showing top 10 rows



In [ ]:
print("find all flights origin from SFO with parameters ")

df_sfo = spark.sql("""
SELECT DayOfWeek, DepDelay, Origin, Dest FROM
us_delay_flights_tbl WHERE origin = 'SFO'"""
).show(10)


find all flights origin from SFO with parameters 
+---------+--------+------+----+
|DayOfWeek|DepDelay|Origin|Dest|
+---------+--------+------+----+
|        6|      -5|   SFO| SLC|
|        7|      -7|   SFO| SLC|
|        1|      -3|   SFO| SLC|
|        2|      NA|   SFO| SLC|
|        3|      -5|   SFO| SLC|
|        4|      -1|   SFO| SLC|
|        5|      75|   SFO| SLC|
|        6|      -2|   SFO| SLC|
|        7|      -9|   SFO| SLC|
|        1|      83|   SFO| SLC|
+---------+--------+------+----+
only showing top 10 rows



In [ ]:
print("\n***** ***** ***** ***** \n")


***** ***** ***** ***** 



In [ ]:
print("find all flights between San Francisco (SFO) and Chicago (ORD) with at least a two-hour delay: ")

spark.sql("""SELECT DayOfWeek, CarrierDelay, Origin, Dest
FROM us_delay_flights_tbl
WHERE CarrierDelay > 120 AND Origin = 'SFO' AND Dest = 'ORD'
ORDER by CarrierDelay DESC""").show(10)

find all flights between San Francisco (SFO) and Chicago (ORD) with at least a two-hour delay: 
+---------+------------+------+----+
|DayOfWeek|CarrierDelay|Origin|Dest|
+---------+------------+------+----+
|        7|         666|   SFO| ORD|
|        4|         426|   SFO| ORD|
|        2|         419|   SFO| ORD|
|        2|         379|   SFO| ORD|
|        7|         376|   SFO| ORD|
|        5|         346|   SFO| ORD|
|        7|         346|   SFO| ORD|
|        6|         336|   SFO| ORD|
|        4|         330|   SFO| ORD|
|        2|         328|   SFO| ORD|
+---------+------------+------+----+
only showing top 10 rows



In [ ]:
print("\n***** ***** ***** ***** \n")



***** ***** ***** ***** 



In [ ]:
print("""
Label all US flights, regardless of origin and destination,
with an indication of the delays they experienced:

add these human-readable labels in a new column called Flight_Delays.
""")



Label all US flights, regardless of origin and destination,
with an indication of the delays they experienced:

add these human-readable labels in a new column called Flight_Delays.



In [ ]:
spark.sql("""SELECT DepDelay, Origin, Dest,
CASE
   WHEN DepDelay > 360 THEN 'Very Long Delays'
   WHEN DepDelay > 120 AND DepDelay < 360 THEN 'Long Delays'
   WHEN DepDelay > 60 AND DepDelay < 120 THEN 'Short Delays'
   WHEN DepDelay > 0 and DepDelay < 60 THEN 'Tolerable Delays'
   WHEN DepDelay = 0 THEN 'No Delays'
ELSE 'Early'
END AS Flight_Delays
FROM us_delay_flights_tbl
ORDER BY Origin, DepDelay DESC""").show(10)


+--------+------+----+-------------+
|DepDelay|Origin|Dest|Flight_Delays|
+--------+------+----+-------------+
|      NA|   SFO| BOI|        Early|
|      NA|   SFO| RNO|        Early|
|      NA|   SFO| FAT|        Early|
|      NA|   SFO| PDX|        Early|
|      NA|   SFO| SBP|        Early|
|      NA|   SFO| ORD|        Early|
|      NA|   SFO| MRY|        Early|
|      NA|   SFO| SNA|        Early|
|      NA|   SFO| SLC|        Early|
|      NA|   SFO| RNO|        Early|
+--------+------+----+-------------+
only showing top 10 rows



In [ ]:
print("\n***** ***** ***** ***** \n")



***** ***** ***** ***** 



In [ ]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import count
from pyspark.sql.types import *

In [ ]:
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
# 9. create an SQL 2
##### ##### ##### ##### ##### ##### ##### ##### ##### #####
spark = (SparkSession
   .builder
   .appName("SparkSQLExampleApp")
   .getOrCreate())

In [ ]:
# Path to data set
print(spark.catalog.listDatabases())
print(spark.catalog.listTables())
spark.catalog.listColumns("us_delay_flights_tbl")


[Database(name='default', catalog='spark_catalog', description='default database', locationUri='file:/content/spark-warehouse')]
[Table(name='us_delay_flights_tbl', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]


[Column(name='Year', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False),
 Column(name='Month', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False),
 Column(name='DayofMonth', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False),
 Column(name='DayOfWeek', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False),
 Column(name='DepTime', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
 Column(name='CRSDepTime', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False),
 Column(name='ArrTime', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
 Column(name='CRSArrTime', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False),
 Column(name='UniqueCarrier', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False)

In [ ]:
df_sfo = spark.sql("""
SELECT DayOfWeek, DepDelay, Origin, Dest FROM
us_delay_flights_tbl WHERE origin = 'SFO'"""
)

In [ ]:
#since 2.2
df_sfo.createOrReplaceGlobalTempView("us_origin_airport_SFO_global_tmp_view")
#df_jfk.createOrReplaceTempView("us_origin_airport_JFK_tmp_view")

In [ ]:
spark.sql("""
SELECT * FROM global_temp.us_origin_airport_SFO_global_tmp_view
""").show(2)


+---------+--------+------+----+
|DayOfWeek|DepDelay|Origin|Dest|
+---------+--------+------+----+
|        6|      -5|   SFO| SLC|
|        7|      -7|   SFO| SLC|
+---------+--------+------+----+
only showing top 2 rows

